# Exercise 2.6: Loading, Inspecting and Cleaning (Angola IEA)

`IEA_2025_IV_TRIM_IND.sav`: the Inquerito ao Emprego em Angola, 4th quarter 2025,
published by INE Angola. 53,353 people, 206 columns, labels in Portuguese.

You will practice: reading the codebook to choose columns, renaming them to
something readable, loading with value labels, recasting what the labels got
wrong, and measuring what is missing.

**PT:** `IEA_2025_IV_TRIM_IND.sav`: Inquerito ao Emprego em Angola, IV trimestre
2025, publicado pelo INE Angola. 53.353 pessoas, 206 colunas, etiquetas em
portugues.

Vai praticar: ler o dicionario de variaveis para escolher colunas, renomea-las
para algo legivel, carregar com etiquetas de valores, corrigir os tipos que as
etiquetas estragaram, e medir o que esta em falta.

> **Pipeline:** reads `0_raw/`, writes `10_cleaned/`.

### Path Setup (run first)

Define the country folder once, then join sub folder and file name onto it.

**PT:** Defina a pasta do pais uma vez e depois junte a subpasta e o nome do
ficheiro.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Country folder first, then join onwards
# Primeiro a pasta do pais, depois juntar o resto
DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_CLEAN_DIR = '../../data/10_cleaned'

EMPLOYMENT_DIR = 'employment_survey'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'
raw_path = os.path.join(DATA_RAW_DIR, EMPLOYMENT_DIR, RAW_FILE)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Data path:', raw_path)
print('Exists?:', os.path.exists(raw_path))

---

## Task 1: Read the codebook

206 columns named `ATW_PAY` or `SRH_AVN` tell you nothing. The file carries a
description for every one, so read that first.

`pd.read_spss` does not expose these descriptions, so this cell uses `pyreadstat`
with `metadataonly=True`, which reads the header without loading any data.

> Just run this cell and the next one. They set up the codebook you will use to
> choose columns in Task 3.

**PT:** 206 colunas com nomes como `ATW_PAY` nao dizem nada. O ficheiro traz uma
descricao de cada uma. `pd.read_spss` nao da acesso a estas descricoes, por isso
esta celula usa `pyreadstat` com `metadataonly=True`, que le o cabecalho sem
carregar dados.

> Basta correr esta celula e a seguinte. Preparam o dicionario que vai usar para
> escolher colunas na Tarefa 3.

In [ ]:
import pyreadstat

# Header only, no data loaded / Apenas o cabecalho, sem carregar dados
_, meta = pyreadstat.read_sav(raw_path, metadataonly=True)
codebook = meta.column_names_to_labels

print('Variables described in the file:', len(codebook))
for name in ['PROV', 'DEM_AGE', 'ATW_PAY', 'SRH_AVN']:
    print(f'{name:12s} {codebook[name][:60]}')

---

## Task 2: Find columns by what they measure

Search the descriptions instead of guessing at names. The number of matches is
itself informative: a precise concept returns one column, a vague one returns
twenty and forces you to read.

**PT:** Pesquise nas descricoes em vez de adivinhar nomes. O numero de
resultados ja e informativo: um conceito preciso devolve uma coluna, um conceito
vago devolve vinte e obriga a ler.

In [ ]:
def find_columns(codebook, keyword):
    """Columns whose description contains `keyword`, case insensitive.

    Colunas cuja descricao contem `keyword`, ignorando maiusculas.
    """
    return {name: label for name, label in codebook.items()
            if label and keyword.lower() in label.lower()}


for keyword in ['sexo', 'idade', 'provincia', 'horas']:
    print(f'{keyword:12s} {len(find_columns(codebook, keyword)):3d} matches')

In [ ]:
# A vague keyword needs reading, not trusting
# Uma palavra vaga precisa de ser lida, nao de confianca cega
for name, label in find_columns(codebook, 'idade').items():
    print(f'{name:16s} {label[:70]}')

**Questions:**

- How many columns match `sexo`? And `idade`? Why the difference?
- Read the `idade` matches. Which is the respondent's own age, and what are the
  others?
- Is searching the codebook enough on its own?

**PT:** Quantas colunas correspondem a `sexo`? E a `idade`? Porque a diferenca?
Qual delas e a idade do proprio inquirido? Chega pesquisar o dicionario?

---

## Task 3: Select the columns and give them readable names

The questionnaire's names are precise and unreadable. `SRH_AVN` is correct and
tells you nothing; `available_last_week` tells you what it holds.

Rename once, here, and every later line of code is easier to check. The mapping
is the bridge back to the official codebook, so it gets saved in Task 4 rather
than living only in this notebook.

**PT:** Os nomes do questionario sao precisos e ilegiveis. `SRH_AVN` esta certo e
nao diz nada; `available_last_week` diz o que contem.

Renomeie uma vez, aqui, e todas as linhas seguintes ficam mais faceis de
verificar. O mapeamento e a ponte de volta ao dicionario oficial, por isso e
gravado na Tarefa 4.

In [ ]:
# Questionnaire name -> readable name / Nome do questionario -> nome legivel
RENAME_MAP = {
    'NIDF': 'household_id',
    'PPNO': 'person_id',
    'G_06_ID_IEA': 'cluster_id',
    'PROV': 'province',
    'AREA_RESID': 'area_type',
    'G_15_TRIMESTRE': 'quarter',
    'DEM_REL': 'relation_to_head',
    'DEM_SEX': 'sex',
    'DEM_AGE': 'age',
    'DEM_MRT': 'marital_status',
    'DEM_EDL': 'education_level',
    'S03_01': 'attended_school',
    'ATW_PAY': 'worked_for_pay',
    'ATW_PFT': 'worked_own_account',
    'ATW_FAM': 'worked_family_business',
    'ABS_JOB': 'absent_from_job',
    'SRH_JOB': 'sought_job',
    'SRH_BUS': 'sought_business',
    'SRH_AVN': 'available_last_week',
    'SRH_AVL': 'available_next_2weeks',
    'SRH_DES': 'wants_work',
    'WKT_USHRSTOT': 'usual_hours',
    'WKT_ACHRSTOT': 'actual_hours',
    'MJT_SYR': 'job_start_year',
    'MJJ_EMP_REL': 'employment_relation',
    'GHVEDT': 'interview_date',
    'POND_IEA_IV_TRIM_2025_IND': 'weight',
}

SPSS_COLS =   # your code here: the keys of RENAME_MAP
              # o seu codigo aqui: as chaves de RENAME_MAP
print('Selected:', len(SPSS_COLS), 'of', len(codebook))

**Questions:**

- How many of the 206 columns did you keep, and why are the household size
  columns not among them?
- `RENAME_MAP` is used twice, for two different purposes. What are they?
- What do you lose by renaming, and how does Task 4 make up for it?

**PT:** Quantas das 206 colunas manteve, e porque as colunas de dimensao do
agregado nao estao la? `RENAME_MAP` e usado duas vezes, para que? O que se perde
ao renomear, e como e que a Tarefa 4 compensa?

---

## Task 4: Load, rename, and save the codebook

`pd.read_spss` applies the file's value labels by default, so coded variables
arrive as readable Portuguese text.

Then save the three way mapping, original name, new name and description, as its
own small file. Six months from now it is the only thing that will tell you what
`wants_work` was called and what question produced it.

**PT:** `pd.read_spss` aplica as etiquetas de valores por omissao, por isso as
variaveis codificadas chegam como texto legivel em portugues.

Depois grave o mapeamento de tres colunas, nome original, nome novo e descricao,
num ficheiro proprio. Daqui a seis meses e a unica coisa que lhe dira como se
chamava `wants_work` e que pergunta a produziu.

In [ ]:
df = pd.read_spss(  # your code here: usecols=SPSS_COLS / o seu codigo aqui )
df = df.  # your code here: rename with RENAME_MAP / renomear com RENAME_MAP

print('Loaded:', df.shape)
df.head()

In [ ]:
# Keep the bridge back to the official documentation
# Guardar a ponte de volta a documentacao oficial
codebook_df = pd.DataFrame({
    'original_name': SPSS_COLS,
    'new_name':   # your code here / o seu codigo aqui
    'description':   # your code here / o seu codigo aqui
})

# Record how each column arrived, so Task 8 can show what had to change
# Registar como cada coluna chegou, para a Tarefa 8 mostrar o que mudou
codebook_df['dtype_loaded'] = [str(df[new_name].dtype)
                               for new_name in codebook_df['new_name']]

os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
codebook_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_codebook.csv')
codebook_df.to_csv(codebook_path, index=False)

print('Saved:', codebook_path)
codebook_df.head(8)

**Questions:**

- How many rows and columns did you load?
- Look at `province` and `sex`. Text or numbers? What decided that?
- Who is the codebook file for, and what would be lost without it?

**PT:** Quantas linhas e colunas carregou? `province` e `sex` sao texto ou
numeros? Para quem e o ficheiro de dicionario, e o que se perderia sem ele?

---

## Task 5: Summary statistics

Look hard at every `min` and `max`, and at any `count` below 53,353.

**PT:** Olhe com atencao para cada `min` e `max`, e para qualquer `count` abaixo
de 53.353.

In [ ]:
df.describe(  # your code here: include='all' / o seu codigo aqui ).T

**Questions:**

- Look at the `max` of `usual_hours`. Is that a possible working week? What is it
  then?
- `age` runs 0 to 120. Which end is real?
- What is `interview_date` really?
- Which columns are answered by only a fifth of the sample, and why?

**PT:** O `max` de `usual_hours` e uma semana possivel? Que extremo de `age` e
real? O que e `interview_date`? Que colunas so um quinto responde, e porque?

---

## Task 6: Explore categories

`value_counts()` shows what is actually in a coded column. Always pass
`dropna=False`. A bar plot makes the shape obvious at a glance.

**PT:** `value_counts()` mostra o que esta realmente numa coluna codificada. Use
sempre `dropna=False`. Um grafico de barras torna a forma imediata.

In [ ]:
print(df['province'].  # your code here: value_counts(dropna=False).sort_index() )

In [ ]:
# Sort before plotting so the bars carry the ranking
# Ordenar antes de desenhar para que as barras mostrem o ranking
df['province'].  # your code here: value_counts, sort_values, plot barh
                # o seu codigo aqui: value_counts, sort_values, plot barh
plt.title('People interviewed by province')
plt.xlabel('People / Pessoas')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
print(df['area_type'].value_counts(dropna=False))
print()
print(df['sex'].value_counts(dropna=False))
print()
print('Distinct households:', df['household_id'].nunique())
print(df['household_id'].value_counts().describe())

**Questions:**

- How many provinces appear, and which is largest?
- Look at the bar plot. Does the sample look proportional to population? What
  does that imply about the `weight` column?
- How many households, and how many people each on average?

**PT:** Quantas provincias aparecem e qual e a maior? O grafico parece
proporcional a populacao? O que isso implica sobre a coluna `weight`? Quantos
agregados, e quantas pessoas em media?

---

## Task 7: Read the dtypes critically

The dtypes decide the next hour of work.

**PT:** Os tipos de dados determinam a proxima hora de trabalho.

In [ ]:
print(type(df))
print(type(df['age']))

In [ ]:
df.  # your code here / o seu codigo aqui

**Questions:**

- Which columns are `category` and which `float64`? What decided that?
- Three dtypes are wrong for what the column means. Find them.

**PT:** Que colunas sao `category` e quais `float64`? Tres tipos estao errados
para o que a coluna significa. Encontre-os.

---

## Task 8: Recast what the file got wrong

`job_start_year` is the year somebody started their main job, and it arrived as a
category. Look at its categories to see why: alongside the years there is a text
label, so pandas concluded the whole column was categorical. A year you cannot
subtract is useless.

`pd.to_numeric` with `errors='coerce'` fixes it in one move: real years convert,
the text label becomes `NaN`. A year is a whole number, so store it as a nullable
integer, and summarise it with the median rather than the mean, since a year has
no meaningful average once part of the column is unknown.

**PT:** `job_start_year` e o ano em que a pessoa comecou o emprego principal, e
chegou como categoria. Veja as categorias: alem dos anos ha uma etiqueta de
texto, por isso o pandas tornou a coluna categorica. Um ano que nao se pode
subtrair nao serve.

`pd.to_numeric` com `errors='coerce'` resolve: os anos reais convertem, a
etiqueta vira `NaN`. Um ano e um numero inteiro, entao guarde como inteiro que
aceita nulos, e resuma com a mediana em vez da media.

In [ ]:
# What is in there besides years? / O que ha alem de anos?
print(df['job_start_year'].value_counts().head(6))

In [ ]:
df['job_start_year'] = (
    # your code here: to_numeric with errors='coerce', then Int64
    # o seu codigo aqui: to_numeric com errors='coerce', depois Int64
)

print('dtype:  ', df['job_start_year'].dtype)
print('range:  ', df['job_start_year'].min(), 'to', df['job_start_year'].max())
print('median: ', df['job_start_year'].median())
print('unknown:', df['job_start_year'].isna().sum())

In [ ]:
# age is a whole number of years with no missing values, so plain int64 works.
# job_start_year above needed the nullable Int64 because it does have gaps.
# age e um numero inteiro de anos sem valores em falta, por isso int64 simples chega.
# job_start_year precisou de Int64 porque tem falhas.
print('age missing values:', df['age'].isna().sum())

df['age'] =   # your code here: int64, no nullable type needed
            # o seu codigo aqui: int64, sem tipo que aceite nulos

print('age dtype:', df['age'].dtype, '| range:', df['age'].min(), 'to', df['age'].max())

In [ ]:
# The hours columns kept the same kind of code, but with no label attached,
# so they stayed numeric and it has to go by hand.
# As colunas de horas mantiveram o mesmo tipo de codigo, mas sem etiqueta,
# por isso continuam numericas e tem de ser tratadas a mao.
print('hours max before:', df['usual_hours'].max())

df['usual_hours'] = df['usual_hours'].  # your code here / o seu codigo aqui
df['actual_hours'] = df['actual_hours'].  # your code here / o seu codigo aqui

print('hours max after: ', df['usual_hours'].max())

In [ ]:
# Identifiers are labels, not quantities: float to integer to string, or the
# trailing .0 survives and joins to nothing.
# Identificadores sao etiquetas, nao quantidades: float para inteiro para texto.
print('Before:', df['household_id'].head(3).tolist())

for col in ['household_id', 'person_id', 'cluster_id']:
    df[col] =   # your code here: int64 then string / inteiro depois texto

print('After: ', df['household_id'].head(3).tolist())

In [ ]:
# interview_date is the float 20251204.0. Int64 tolerates the missing values.
# interview_date e o float 20251204.0. Int64 aceita os valores em falta.
df['interview_date'] = pd.to_datetime(
    # your code here: Int64 then string, format='%Y%m%d'
    # o seu codigo aqui: Int64 depois texto, format='%Y%m%d'
)

print('dtype:', df['interview_date'].dtype)
print('Range:', df['interview_date'].min(), 'to', df['interview_date'].max())
print()
print(df['interview_date'].dt.month.value_counts(dropna=False).sort_index())

In [ ]:
# Record the corrected type next to the original one, then re-save the codebook.
# It now documents which columns needed fixing and what they became.
# Registar o tipo corrigido ao lado do original e regravar o dicionario.
codebook_df['dtype_final'] =   # your code here: the dtype of each column now
                              # o seu codigo aqui: o tipo atual de cada coluna

changed = codebook_df[codebook_df['dtype_loaded'] != codebook_df['dtype_final']]
print('Columns whose type changed / Colunas cujo tipo mudou:', len(changed))
print(changed[['new_name', 'dtype_loaded', 'dtype_final']].to_string(index=False))

codebook_df.to_csv(codebook_path, index=False)

**Questions:**

- Run `value_counts()` on `job_start_year`. Which value is not a year, and why
  did one such value change the whole column's dtype?
- After the cast, what is the median and how many values are unknown? Why report
  the median rather than the mean of a year?
- `age` becomes `int64` but `job_start_year` becomes `Int64`. Why the difference?
- Why did the hours columns need a manual replacement when `job_start_year` did
  not?
- What does the identifier look like if you skip the `int64` step?
- Look at the date range and the month counts. Is this really the 4th quarter?
  What is missing entirely?

**PT:** Que categoria de `job_start_year` nao e um ano, e porque mudou o tipo da
coluna toda? Qual e a mediana e quantos valores sao desconhecidos? Porque a
mediana e nao a media? Porque as colunas de horas precisaram de tratamento
manual? Isto e mesmo o IV trimestre? O que falta por completo?

---

## Task 9: Detect missing values

Count them, express them as a share, and look at the shape before deciding
anything.

**PT:** Conte, converta em percentagem, e observe o padrao antes de decidir.

In [ ]:
missing = pd.DataFrame({
    'n_missing':   # your code here / o seu codigo aqui
    'pct_missing':   # your code here, rounded to 1 / arredondado a 1
})
missing.sort_values('pct_missing', ascending=False)

In [ ]:
counts = df.isna().sum()
counts[counts > 0].sort_values().plot(kind='barh', color='coral', figsize=(9, 6))
plt.title('Missing values by column')
plt.xlabel('Count / Contagem')
plt.tight_layout()
plt.show()

**Questions:**

- Which columns are most missing? Is that damage, or something else?
- What decides whether a gap is a problem?
- What would `dropna()` do to this dataset, and why?

**PT:** Que colunas tem mais valores em falta? E dano ou outra coisa? O que
decide se uma falha e um problema? O que faria `dropna()` a este conjunto?

---

## Task 10: Save

Raw data is read only. Write to `10_cleaned/` and reload to confirm the round
trip.

**PT:** Os dados brutos sao apenas de leitura. Grave em `10_cleaned/` e
recarregue para confirmar.

In [ ]:
out_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')

df.  # your code here: to_csv with index=False / to_csv com index=False
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype={'household_id': 'string',
                                     'person_id': 'string',
                                     'cluster_id': 'string'})
print('Reloaded:', check.shape)
print(check[['household_id', 'province', 'interview_date', 'job_start_year']].dtypes)

**Questions:**

- How many rows does the saved file have? Was anything removed?
- Reload it and check the dtypes. Which did not survive, and why?
- Which artefact from this notebook does survive the round trip intact?

**PT:** Quantas linhas tem o ficheiro gravado? Foi removido alguma coisa? Que
tipos nao sobreviveram ao recarregar, e porque? Que artefacto deste caderno
sobrevive intacto?